Імпорти

In [2]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [47]:
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
    TEXT_COLUMN,
    TARGET_COLUMN,
    MODELS_DIR,
    MODEL_RESULTS_PATH,
)

from src.evaluation import save_experiment_result

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(device)

mps


1. Завантажуємо дані

In [5]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (7999, 4)
Validation: (2000, 4)
Test: (3080, 4)


In [6]:
BERT_TEXT_COLUMN = "text"

required_columns = {BERT_TEXT_COLUMN, TARGET_COLUMN}

for name, frame in {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}.items():
    missing = required_columns - set(frame.columns)

    if missing:
        raise ValueError(
            f"{name} dataset is missing columns: {missing}"
        )

2. Перетворюємо Pandas у Hugging Face Dataset

In [7]:
def prepare_bert_frame(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df[[BERT_TEXT_COLUMN, TARGET_COLUMN]]
        .rename(
            columns={
                BERT_TEXT_COLUMN: "text",
                TARGET_COLUMN: "labels",
            }
        )
        .reset_index(drop=True)
    )

In [8]:
hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(
        prepare_bert_frame(train_df),
        preserve_index=False,
    ),
    "validation": Dataset.from_pandas(
        prepare_bert_frame(val_df),
        preserve_index=False,
    ),
    "test": Dataset.from_pandas(
        prepare_bert_frame(test_df),
        preserve_index=False,
    ),
})

hf_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 7999
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 3080
    })
})

3. Label mappings

In [9]:
label_mapping_df = (
    train_df[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
)

id2label = dict(
    zip(
        label_mapping_df["label"].astype(int),
        label_mapping_df["label_text"],
    )
)

label2id = {
    label_name: label_id
    for label_id, label_name in id2label.items()
}

NUM_LABELS = len(id2label)

print("Number of labels:", NUM_LABELS)
print(list(id2label.items())[:5])

Number of labels: 77
[(0, 'activate_my_card'), (1, 'age_limit'), (2, 'apple_pay_or_google_pay'), (3, 'atm_support'), (4, 'automatic_top_up')]


4. Робимо Tokenizer

In [10]:
MODEL_CHECKPOINT = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT
)

In [11]:
MAX_LENGTH = 128

In [12]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

In [13]:
tokenized_dataset = hf_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

Map:   0%|          | 0/7999 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [14]:
tokenized_dataset["train"][0]

{'labels': 38,
 'input_ids': [101, 2129, 2079, 1045, 5227, 1996, 9231, 1029, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

5. Робимо Dynamic padding

In [15]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

6. Робимо функцію для обчислення метрик

In [16]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "macro_precision": precision_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

7. Ініціюємо модель

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


8. Конфігуруємо параметри навчання

In [18]:
DISTILBERT_OUTPUT_DIR = (
    MODELS_DIR / "distilbert-banking-intent"
)

DISTILBERT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [22]:
training_args = TrainingArguments(
    output_dir=str(DISTILBERT_OUTPUT_DIR),

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    fp16=torch.cuda.is_available(),
    bf16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
    seed=42,
    data_seed=42,
)

9. Створюємо Trainer

In [23]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ],
)

10. Навчання

In [24]:
start_time = time.perf_counter()

train_result = trainer.train()

distilbert_train_time = (
    time.perf_counter() - start_time
)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall
1,0.542897,0.558518,0.884000,0.869081,0.880334,0.879874,0.870852
2,0.333812,0.422490,0.894500,0.886693,0.893175,0.902304,0.884984
3,0.268470,0.379822,0.906500,0.900707,0.905698,0.913445,0.898439


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
print(
    "Training time:",
    round(distilbert_train_time / 60, 2),
    "minutes",
)

Training time: 7.01 minutes


11. Зробимо оцінку на validation

In [26]:
validation_metrics = trainer.evaluate(
    tokenized_dataset["validation"]
)

validation_metrics

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,Macro Precision,Macro Recall
0.268470,0.379822,3,0.906500,0.900707,0.905698,0.913445,0.898439


{'eval_loss': 0.3798219561576843,
 'eval_accuracy': 0.9065,
 'eval_macro_f1': 0.900707484621432,
 'eval_weighted_f1': 0.905697570084648,
 'eval_macro_precision': 0.9134452627507342,
 'eval_macro_recall': 0.898439229219944}

12. Зберігаємо найкращу модель

In [27]:
FINAL_MODEL_DIR = (
    MODELS_DIR / "distilbert-banking-intent-final"
)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/Users/oleh/PycharmProjects/banking-intent-classification/models/distilbert-banking-intent-final/tokenizer_config.json',
 '/Users/oleh/PycharmProjects/banking-intent-classification/models/distilbert-banking-intent-final/tokenizer.json')

13. Додаємо результати до таблиці експериментів

In [48]:
distilbert_result = {
    "model": "DistilBERT",
    "variant": "fine-tuned",
    "split": "validation",
    "parameters": {
        "checkpoint": MODEL_CHECKPOINT,
        "max_length": MAX_LENGTH,
        "learning_rate": 2e-5,
        "train_batch_size": 16,
        "eval_batch_size": 32,
        "epochs": 3,
        "weight_decay": 0.01,
    },
    "accuracy": validation_metrics["eval_accuracy"],
    "macro_f1": validation_metrics["eval_macro_f1"],
    "weighted_f1": validation_metrics["eval_weighted_f1"],
    "macro_precision": validation_metrics[
        "eval_macro_precision"
    ],
    "macro_recall": validation_metrics[
        "eval_macro_recall"
    ],
    "train_time_sec": distilbert_train_time,
    "n_features": None,
}

Збережемо окремо результати

In [49]:
results_df = save_experiment_result(
    distilbert_result,
    MODEL_RESULTS_PATH,
)

results_df

,model,parameters,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,train_time_sec,variant,split,n_features
0,DistilBERT,"{""checkpoint"": ""distilbert-base-uncased"", ""epo...",0.9065,0.900707,0.905698,0.913445,0.898439,420.774839,fine-tuned,validation,None
1,TF-IDF + Linear SVM,"{""classifier__C"": 1.0, ""tfidf__min_df"": 1, ""tf...",0.8790,0.873237,0.878779,0.881031,0.871352,3.870460,tuned,validation,2158.0
2,TF-IDF + Linear SVM,"{""C"": 1.0, ""min_df"": 2, ""ngram_range"": [1, 2],...",0.8755,0.870909,0.875092,0.879338,0.868093,0.367386,baseline,validation,8913.0
3,TF-IDF + Logistic Regression,"{""classifier__C"": 2.0, ""tfidf__min_df"": 2, ""tf...",0.8675,0.862007,0.867527,0.874387,0.858843,15.551374,tuned,validation,1331.0
4,TF-IDF + Logistic Regression,"{""C"": 1.0, ""min_df"": 2, ""ngram_range"": [1, 2],...",0.8410,0.829135,0.839451,0.854706,0.824639,2.555273,baseline,validation,8913.0
5,DummyClassifier,"{""strategy"": ""most_frequent""}",0.0190,0.000484,0.000709,0.000247,0.012987,0.001041,baseline,validation,NaN
